# 5장. 파일 입력과 결과물 다운로드

CSV를 sandbox 파일로 공급하고 보고서를 내려받습니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | 파일 입력·Artifact 게시·다운로드 |
| 이번 실습 조합 | OpenAI-hosted · 표준 라이브러리 · 기본 집계와 보고서 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

02–04장의 메시지 기반 CSV 입력을 파일 입력으로 바꿉니다. 입력 메시지에는 CSV 원문 대신 작업 지시와 파일 경로를 전달합니다. 파일을 만들었다는 최종 답변만으로 게시·다운로드·내용의 정확성을 판단하지 않습니다.

| 단계 | 직접 사용하는 API 또는 값 | 확인할 내용 |
| --- | --- | --- |
| 입력 준비 | `environment.files` | 로컬 CSV를 배치할 sandbox 경로 |
| 실행 | `sessions.create()` | Session·Turn ID, 완료 상태, 최종 JSON |
| 게시 확인 | `artifacts.list()` | 이번 Turn의 파일 ID·경로·크기 |
| 다운로드 | `for_result(result).download()` | 선택한 파일과 로컬 저장 경로 |
| 로컬 읽기 | 내려받은 JSON·Markdown | 실제 파일의 분석값·보고서 |
| 정리 | 정리 helper의 삭제·재조회 | 생성한 세션의 API상 부재 |

수동 실습은 아래 셀을 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_intro.py --chapter 5 --live`로 실행합니다. 수동 출력과 별도 검증기의 통과 기록을 구분하세요.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실습 준비와 실행 모드

로컬에서는 저장소 루트에서 `uv sync --locked`, `uv run jupyter lab`을 실행하고 `.venv` 커널을 선택하세요.
Colab에서는 다운로드 안내의 독립 실행본을 업로드하세요. 준비 셀이 패키지를 직접 설치하고 포함된 코드·데이터를 생성합니다.
실제 API 실습은 **RUN_LIVE=True**로 명시적으로 허용합니다.
키는 `.env.local` 또는 Colab Secrets에서 읽으며 출력하지 않습니다. 라이브 실행에는 비용이 발생합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
NUMBER = 5
CHAPTER = "05_files_artifacts"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print(
    {
        "chapter": NUMBER,
        "live_enabled": settings.enabled,
        "model": settings.model or "Offline mode",
    }
)


## 1. 입력 파일과 경로 준비하기

| 위치 | 이 실습의 경로 | 역할 |
| --- | --- | --- |
| 로컬 입력 | `data/sales.csv` | 노트북이 읽는 원본 |
| sandbox 입력 | `/workspace/sales.csv` | 에이전트가 읽는 파일 |
| sandbox 출력 | `/workspace/outputs/summary.json`, `/workspace/outputs/report.md` | 에이전트가 작성하는 결과 |
| 로컬 다운로드 | `outputs/intro/<chapter>/<run_id>/downloads/` | 사용자가 보관하는 결과 |

`environment.files[].path`는 sandbox에 배치할 경로입니다. 나중에 사용할 `download(..., to=...)`의 목적지는 로컬 경로입니다. 같은 파일 이름이어도 다른 파일시스템의 파일입니다.

먼저 로컬 CSV를 바이트로 읽고 내용을 표시합니다. 이 단계에서는 API를 호출하거나 파일을 전송하지 않습니다.


In [ ]:
input_path = settings.root / "data/sales.csv"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
print(csv_text)


### inline 파일 입력

base64는 파일 바이트를 JSON 요청에 담기 위한 인코딩입니다. `data`에는 일반 base64 문자열을 전달하며 data URL 접두사는 붙이지 않습니다. 긴 인코딩 문자열 대신 입력 경로·바이트 수·출력 경로를 표시합니다.

이 방식은 별도의 Files API 업로드 개체나 file ID를 만들지 않습니다. 환경 설정은 다음 실행 단계의 `sessions.create()`에서 전송됩니다. 표준 라이브러리만 사용하므로 sandbox 외부 네트워크는 끕니다.


In [ ]:
import base64

environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
    "files": [{
        "type": "inline",
        "path": "/workspace/sales.csv",
        "data": base64.b64encode(csv_bytes).decode("ascii"),
    }],
}
print({
    "local_input": str(input_path),
    "sandbox_input": environment["files"][0]["path"],
    "input_bytes": len(csv_bytes),
    "sandbox_outputs": ["/workspace/outputs/summary.json", "/workspace/outputs/report.md"],
})


### 파일을 읽고 결과를 저장하도록 요청하기

입력 메시지는 기존 `/workspace/sales.csv`를 Python으로 읽고 두 결과 파일을 만들도록 지시합니다. `METRICS_CONTRACT`를 실제 메시지에 조합해 최종 JSON 형식을 한 곳에서 정의합니다. 보고서는 한국어로 작성하도록 요청하며 영어 교재에서도 동일한 실행 코드를 사용합니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)
task = (
    "Use Python's standard csv module to read /workspace/sales.csv and compute the summary. "
    "Write /workspace/outputs/summary.json containing exactly the requested summary. "
    "Also write /workspace/outputs/report.md in Korean, starting with '# 매출 분석 보고서'. "
    "Include row count, total quantity, total revenue as digits, and every product's revenue. "
    "Read both files back and verify they exist. "
    + METRICS_CONTRACT
)


## 2. 에이전트 실행과 완료 확인

파일 입력을 담은 환경·Agent 지침·작업 메시지를 세션 생성 요청에 직접 전달합니다. 스트리밍 텍스트와 최종 JSON은 모델의 응답입니다. 파일 게시·다운로드와는 별도로 확인합니다.

각 단계는 오류·일반적인 인터럽트에서 `cleanup_on_error()`로 정리를 시도합니다. 정상 경로에서는 **6절 정리 셀까지 실행**하고, 중간에 학습을 멈출 때도 정리 셀을 실행하세요. 강제 커널 종료에서는 자동 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

`output_dir`는 이번 실행의 복구 기록과 다운로드를 보관하는 로컬 경로입니다. 상태 준비 셀은 새 실행 시작에 한 번만 실행하세요. 진행 중 재실행하면 자원 식별자를 잃을 수 있습니다.

타이머는 아래 API 실행 직전에 시작합니다. 600초 제한은 이벤트 도착 시 확인하므로 무응답 중 강제 중단하는 전체 타이머는 아닙니다. 클라이언트 HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import json
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/intro" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
downloads = output_dir / "downloads"
downloads_complete = False


### 세션 생성과 스트리밍

`agent.session.created`에서 Session ID를 즉시 기록하고 텍스트 delta를 표시합니다. 예상하지 않은 외부 작업은 오류로 처리하고 `check_event()`로 실패·취소를 감지합니다. 이벤트 수·순서를 고정하거나 `idle`만으로 성공을 판단하지 않습니다.

`with_result_collection()`으로 모은 응답을 `get_final_result()`로 꺼내고 저장된 root Turn도 조회합니다. 출력에서 완료 상태와 최종 JSON을 확인하세요. 이 셀은 새 작업을 제출하므로 같은 실행 상태에서 중복 제출은 차단합니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})

    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nCompleted analysis:")
    print({"session_id": result.session_id, "turn_id": result.turn_id, "status": turn.status})
    print(json.dumps(json.loads(result.output_text), indent=2))


## 3. 이번 Turn에서 게시된 파일 확인하기

| 대상 | 의미 |
| --- | --- |
| sandbox 파일 | 에이전트가 읽고 쓰는 작업 파일 |
| artifact | 완료한 Turn에서 게시한 불변 사본 |
| 로컬 다운로드 | 사용자가 보관하고 세션 삭제 후에도 읽는 파일 |

`/workspace/outputs`에 작성한 결과의 게시 목록을 확인합니다. `artifacts.list(session_id)`는 세션 전체의 목록이므로 이번 결과의 Session·Turn ID에 맞는 항목만 표시합니다. SDK 반복으로 다음 페이지까지 읽습니다.

아래 출력의 파일 ID·경로·Turn ID·바이트 수는 게시 정보입니다. 파일 내용의 정확성이나 로컬 다운로드 완료를 증명하지는 않습니다. 필요한 파일은 세션을 삭제하기 전에 내려받습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting published files.")
    published = list(client.beta.agents.sessions.artifacts.list(result.session_id, timeout=15))
    current_artifacts = [a for a in published
                         if a.session_id == result.session_id and a.turn_id == result.turn_id]
    print("Files published by this turn:")
    print(json.dumps([
        {"id": a.id, "path": a.path, "turn_id": a.turn_id, "bytes": a.size_bytes}
        for a in current_artifacts
    ], indent=2))


## 4. 정확한 Turn·경로로 로컬에 다운로드하기

| 호출 | 목적 |
| --- | --- |
| `artifacts.list()` | 게시된 파일 목록 확인 |
| `artifacts.for_result(result)` | 결과의 Session·Turn으로 선택 범위 고정 |
| `scoped.download(path, to=...)` | 정확한 원격 경로를 골라 로컬 파일로 저장 |

공식 SDK가 Session·Turn·경로가 일치하는 artifact를 찾으며 없거나 여러 개면 오류를 발생시킵니다. 별도의 custom 선택 helper를 만들지 않습니다. 다운로드가 반환한 artifact 정보와 실제 로컬 경로·파일 크기를 함께 표시합니다.

이 셀은 작업을 새로 제출하지 않습니다. `exist_ok=True`로 같은 실행의 다운로드 폴더를 다시 사용할 수 있지만, 재실행은 세션 정리 전으로 한정하세요. 전송 중 오류가 나면 정리를 시도하므로 부분 파일이 남아도 성공으로 취급하지 않습니다. 기본 예제는 다운로드 실패를 숨기거나 분석을 자동 재실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before downloading artifacts.")
    downloads_complete = False
    downloads.mkdir(exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(result)
    summary_artifact = scoped.download(
        "/workspace/outputs/summary.json", to=downloads / "summary.json", timeout=30,
    )
    report_artifact = scoped.download(
        "/workspace/outputs/report.md", to=downloads / "report.md", timeout=30,
    )
    downloads_complete = True
    for artifact, name in [(summary_artifact, "summary.json"), (report_artifact, "report.md")]:
        local_path = downloads / name
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path),
               "local_bytes": local_path.stat().st_size})


## 5. 내려받은 실제 파일 읽기

최종 응답 JSON은 2절에서 확인했습니다. 이제 로컬 `summary.json`을 파싱하고 `report.md`를 표시합니다. 아래 정상 경로는 새 API 요청 없이 로컬 파일을 읽습니다. JSON 문법 확인과 사람이 읽는 출력이며 자동 정답 검사는 아닙니다.

| 확인 | 의미 |
| --- | --- |
| Turn 완료 | 작업 상태 확인 |
| artifact 목록 | 게시 정보 확인 |
| 다운로드 완료와 로컬 읽기 | 결과 파일을 실제로 받아 읽었는지 확인 |
| 별도 검증기 | Python 명령·CSV 기준값·보고서 내용 대조 |

수동 실습에서는 최종 응답과 파일의 분석값을 비교하고 보고서의 제목·행 수·수량·매출을 읽어 보세요. 자동 정답 검사와 검증 기록은 실행 사본에서만 추가합니다. 정상 다운로드한 로컬 파일은 세션 삭제 후에도 읽을 수 있습니다.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete:
        raise RuntimeError("Complete both downloads before reading local results.")
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    report_text = (downloads / "report.md").read_text(encoding="utf-8")
    print("Downloaded summary.json:")
    print(json.dumps(downloaded_summary, indent=2))
    display(Markdown(report_text))


## 6. 세션을 정리하고 로컬 파일 보관하기

필요한 파일을 확보했으므로 이번 실행의 세션을 정리합니다. 아래 셀은 분석이나 다운로드를 다시 실행하지 않습니다. 중간에 학습을 멈출 때도 실행하세요.

`cleanup_resources()`는 이번 실행의 metadata로 생성 응답을 놓친 세션도 찾고, 필요하면 취소 후 삭제·재조회 `404`로 API상 부재를 확인합니다. 출력의 `deleted`·`verified_absent`를 확인하세요. 물리적 sandbox 정리의 동기 완료를 뜻하지는 않습니다.

정리 오류는 원래 실행·검증 오류를 가리지 않도록 처리합니다. 검증 도구는 삭제 전에 이력·파일 정답 검사를 추가하며 실패해도 정리를 시도합니다. 이 장은 별도 저장 Agent나 Files API 업로드 개체를 만들지 않습니다. 그런 개체를 추가하는 예제에서는 보존·삭제를 별도로 관리해야 합니다.

세션 삭제 후 서버 파일을 다시 받는 것에 의존하지 마세요. 로컬 다운로드와 결과 객체는 남으며, 로컬 파일을 지우는 것은 세션 삭제와 별개입니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 7. 실패 시 조치와 다음 장

| 상황 | 할 일 |
| --- | --- |
| 401/403 또는 429 | 인증·프로젝트 권한 또는 API 제한 확인 |
| Turn 완료, 예상 artifact 없음 | 이번 Turn·정확한 원격 경로와 게시 목록 확인. 완료와 파일 게시를 구분 |
| 첫 파일만 다운로드됨 | 두 번째 전송 오류를 성공으로 처리하지 않음. 부분 파일과 정리 결과 확인 |
| 파일은 있지만 전송 오류 발생 | 파일 존재만으로 완료 판단 금지. 기본 오류 처리는 세션 정리를 시도함 |
| 연결 종료·강제 커널 종료 | 저장된 Session ID·정리 결과와 `outputs/intro/<chapter>/<run_id>/creation.json` 확인 |

키나 원본 오류 body를 출력하지 마세요. 같은 입력으로 새 세션을 무조건 만들지 않습니다. 검증 도구로 실행했다면 로컬 `summary.json`도 참고하세요. 자세한 복구 절차는 [빠른 시작](../quickstart.ipynb)을 참고하세요.

수동 실습의 완료 기준은 최종 JSON·게시 목록·로컬 JSON/보고서·세션 삭제 결과를 구분해 읽는 것입니다. [6장](06_environment.ipynb)에서는 같은 파일 흐름에 pandas 설치와 환경 초기화 검사를 추가합니다.

**심화 참고:** [15장 Multi-agent](15_multi_agent.ipynb)는 담당별 파일이 서로 다른 Turn에서 게시될 수 있습니다. 이 장의 `for_result(result)`는 한 완료 Turn에 범위를 고정합니다. 여러 Turn에서 같은 경로를 사용하면 소유 Turn과 파일 버전을 함께 구분해야 합니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Files and artifacts](https://developers.openai.com/api/docs/guides/agents-api/environments/files) — 파일 입력·artifact 다운로드·파일 수명.
- [Manage sessions](https://developers.openai.com/api/docs/guides/agents-api/sessions/manage) — 현재 상태·pending action 확인과 세션 삭제.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
